# AI Shield: CF Detector Freeze v1

This notebook records the exact fine-tuned detector and checks that reusable inference reproduces six saved validation predictions. It does not train models or select new thresholds.

**Run:** Open in Colab, preferably select a T4 GPU, then use **Runtime > Run all** and authorize your Drive mount. CPU also works but is slower. Run in a fresh runtime.

**Inputs:** the existing `finetune_v1` experiment, `best.pt`, and the original Dataset v1 validation images.
**Outputs:** a separate `AI_Shield_CommunityForensics/detector_freezes/cf_finetune_v1_max_f1/` directory.

A completed run must end in **PASS**. A saved hash alone is not a successful inference check. Original images, training results, and checkpoints are read-only.

[Training notebook](https://colab.research.google.com/drive/1q5CB7k9fdY9ylfqMDQgClVl4k_ogNJef) | [Experiment folder](https://drive.google.com/drive/folders/1vcrWLqMnctbe9axz0qVeFuY4g72OShpj) | [Selected checkpoint](https://drive.google.com/file/d/1rE9pc32i-BCijkQ0kJdqQJC9ioRi3RzZ/view)


## 1. Install inference dependencies

Use the recorded timm and Pillow versions. Keep Colab's existing compatible PyTorch/torchvision installation and record its versions; a different runtime must still pass the score comparison. No pretrained weights are downloaded: the selected checkpoint contains the full model state.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "timm==1.0.15", "Pillow==11.3.0", "pandas>=2,<3",
])


## 2. Mount Drive and locate the saved experiment

The default threshold is the saved max-F1 value. The alternative remains recorded for reference. Paths below match the completed training notebook.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
EXPERIMENT = DRIVE_ROOT / "AI_Shield_CommunityForensics/experiments/finetune_v1"
DATA_ROOT = DRIVE_ROOT / "AI_Shield_Data"
FREEZE_ROOT = DRIVE_ROOT / "AI_Shield_CommunityForensics/detector_freezes/cf_finetune_v1_max_f1"

CHECKPOINT = EXPERIMENT / "checkpoints/stage2_last_blocks/best.pt"
MANIFEST_PATH = EXPERIMENT / "experiment_manifest.json"
REPORT_PATH = EXPERIMENT / "results/final_report.json"
THRESHOLDS_PATH = EXPERIMENT / "results/validation_thresholds.json"
PREDICTIONS_PATH = EXPERIMENT / "predictions/finetuned_validation.csv"
SPLIT_PATH = DATA_ROOT / "splits/v1/validation.csv"

DEFAULT_THRESHOLD = 0.6339887976646423
ALTERNATIVE_THRESHOLD = 0.7482541799545288
PROB_ATOL, PROB_RTOL = 1e-5, 1e-5
LOGIT_ATOL, LOGIT_RTOL = 1e-4, 1e-5
SAMPLES_PER_CLASS = 3

for path in (CHECKPOINT, MANIFEST_PATH, REPORT_PATH, THRESHOLDS_PATH,
             PREDICTIONS_PATH, SPLIT_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Required input missing: {path}")
print("All required experiment files found.")


## 3. Cross-check saved records and calculate the fine-tuned hash

This calculates SHA-256 from your actual Stage 2 `best.pt`. The original pretrained weights hash is retained separately. A first hash records identity; it does not independently certify a file's history.

The validation split must still match the split recorded during training.


In [ ]:
import hashlib
import importlib.metadata
import json
import math
from datetime import datetime, timezone
import uuid
import numpy as np
import pandas as pd

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def json_text(value):
    return json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n"

def write_once(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        require(path.read_text(encoding="utf-8") == content,
                f"Existing freeze file differs: {path}. Investigate; do not overwrite it.")
    else:
        with path.open("x", encoding="utf-8") as handle:
            handle.write(content)

experiment = read_json(MANIFEST_PATH)
report = read_json(REPORT_PATH)
thresholds = read_json(THRESHOLDS_PATH)
expected_model = "OwensLab/commfor-model-384"
expected_revision = "6076002bf0d9dd37537f965ee2f06f826c333b61"
for record in (experiment["pretrained_model"], report["pretrained_model"]):
    require(record["model_id"] == expected_model, "Unexpected base model.")
    require(record["revision"] == expected_revision, "Unexpected base-model revision.")
require(report["selected_finetuned_candidate"] == "last_blocks", "Wrong selected stage.")
require(Path(report["selected_checkpoint"]) == CHECKPOINT, "Selected checkpoint path differs.")
require(report["run_id"] == "finetune_v1", "Unexpected experiment ID.")
require(thresholds == report["validation"]["thresholds"], "Threshold files disagree.")
require(thresholds["finetuned"]["max_f1"] == DEFAULT_THRESHOLD, "Max-F1 threshold differs.")
require(thresholds["finetuned"]["fpr_target_threshold"] == ALTERNATIVE_THRESHOLD,
        "Alternative threshold differs.")
require(experiment["preprocessing"]["eval"] ==
        "exif_transpose_rgb_resize440_bilinear_centercrop384_imagenet",
        "Preprocessing record differs.")

checkpoint_sha256 = sha256_file(CHECKPOINT)
manifest_sha256 = sha256_file(MANIFEST_PATH)
split_sha256 = sha256_file(SPLIT_PATH)
require(split_sha256 == experiment["split_csv_sha256"]["validation"],
        "Validation split changed since training.")
print("Fine-tuned best.pt SHA-256:", checkpoint_sha256)
print("Default: max-F1 =", DEFAULT_THRESHOLD)
print("Alternative: validation FPR target =", ALTERNATIVE_THRESHOLD)


## 4. Pin six validation fixtures and record the freeze

Select three real and three generated images in stable sample-ID order. Join predictions to the original split by ID and check labels, paths, and image hashes. These are implementation checks, not a new accuracy benchmark.

The immutable `detector_lock.json` pins the checkpoint, preprocessing, thresholds, source-file hashes, and fixture identities. Re-running with changed inputs stops rather than silently replacing that record.


In [ ]:
predictions = pd.read_csv(PREDICTIONS_PATH, dtype={"sample_id": str, "local_path": str},
                          keep_default_na=False)
split = pd.read_csv(SPLIT_PATH, dtype={"sample_id": str, "local_path": str, "sha256": str},
                    keep_default_na=False)
require({"sample_id", "local_path", "label", "split", "prob_fake", "logit"}
        <= set(predictions.columns), "Prediction columns are missing.")
require({"sample_id", "local_path", "label", "split", "sha256"} <= set(split.columns),
        "Split columns are missing.")
for table in (predictions, split):
    require(len(table) == 640, "Expected 640 validation rows.")
    require(not table["sample_id"].duplicated().any(), "Duplicate sample IDs.")
    require(set(table["split"]) == {"validation"}, "Expected validation data only.")
    require(table["label"].isin([0, 1]).all(), "Unexpected class labels.")
require(set(predictions["sample_id"]) == set(split["sample_id"]), "Image ID sets differ.")
require(np.isfinite(predictions[["prob_fake", "logit"]].to_numpy(dtype=float)).all(),
        "Saved predictions contain non-finite scores.")
require(predictions["prob_fake"].between(0, 1).all(), "Invalid saved probability.")

joined = predictions.merge(
    split[["sample_id", "label", "local_path", "sha256"]],
    on="sample_id", suffixes=("", "_split"), validate="one_to_one",
)
require((joined["label"] == joined["label_split"]).all(), "Labels disagree.")
require((joined["local_path"] == joined["local_path_split"]).all(), "Image paths disagree.")
parts = []
for label in (0, 1):
    part = joined[joined["label"] == label].sort_values("sample_id").head(SAMPLES_PER_CLASS)
    require(len(part) == SAMPLES_PER_CLASS, f"Too few examples for label {label}.")
    parts.append(part)
selected = pd.concat(parts, ignore_index=True)

fixtures = []
for row in selected.to_dict(orient="records"):
    path = Path(row["local_path"])
    require(path.is_file(), f"Validation image missing: {path}")
    actual_hash = sha256_file(path)
    require(actual_hash == row["sha256"], f"Image contents changed: {path}")
    fixtures.append({
        "sample_id": row["sample_id"], "label": int(row["label"]),
        "path": str(path), "sha256": actual_hash,
        "expected_prob_fake": float(row["prob_fake"]),
        "expected_logit": float(row["logit"]),
    })

lock = {
    "schema_version": 1, "detector_id": "cf_finetune_v1_max_f1",
    "checkpoint": {"path": str(CHECKPOINT), "sha256": checkpoint_sha256,
                   "size_bytes": CHECKPOINT.stat().st_size,
                   "drive_file_id": "1rE9pc32i-BCijkQ0kJdqQJC9ioRi3RzZ",
                   "state_dict_key": "model_state"},
    "base_model": experiment["pretrained_model"],
    "architecture": "vit_small_patch16_384.augreg_in21k_ft_in1k",
    "head_outputs": 1, "labels": {"0": "real", "1": "generated"},
    "score": "sigmoid(logit)", "decision_operator": ">=",
    "default_operating_point": "validation_max_f1",
    "default_threshold": DEFAULT_THRESHOLD,
    "alternative_threshold": ALTERNATIVE_THRESHOLD,
    "alternative_policy": "maximize validation recall with FPR <= 0.05",
    "preprocessing": {
        "frame": 0, "exif_transpose": True, "color_mode": "RGB",
        "resize_shorter_side": 440, "interpolation": "bilinear", "antialias": True,
        "center_crop": [384, 384], "tensor_scale": "uint8 / 255",
        "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225],
        "dtype": "float32", "tf32": False,
    },
    "training_libraries": experiment["libraries"],
    "source_sha256": {
        "experiment_manifest": manifest_sha256, "validation_split": split_sha256,
        "validation_predictions": sha256_file(PREDICTIONS_PATH),
        "validation_thresholds": sha256_file(THRESHOLDS_PATH),
        "final_report": sha256_file(REPORT_PATH),
    },
    "comparison": {"prob_atol": PROB_ATOL, "prob_rtol": PROB_RTOL,
                   "logit_atol": LOGIT_ATOL, "logit_rtol": LOGIT_RTOL,
                   "require_same_default_classification": True},
    "fixtures": fixtures,
}
LOCK_PATH = FREEZE_ROOT / "detector_lock.json"
write_once(LOCK_PATH, json_text(lock))
print("Identity recorded; inference verification is still pending.")
display(pd.DataFrame(fixtures)[["sample_id", "label", "expected_prob_fake"]])


## 5. Load the checkpoint with reusable standalone inference

This module uses the exact evaluation transforms. It loads `model_state` strictly, uses float32 evaluation, disables gradients and TF32, and downloads no model weights.

A local snapshot is hashed again before loading so it must match the recorded Drive checkpoint. The checkpoint's internal experiment-manifest hash and stage are checked too.


In [ ]:
MODULE_SOURCE = "\"\"\"Frozen Community Forensics inference, matching finetune_v1 evaluation.\"\"\"\nimport hashlib\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nimport timm\nfrom PIL import Image, ImageOps\nfrom torchvision import transforms\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\nclass CommunityForensicsDetector(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.vit = timm.create_model(\n            \"vit_small_patch16_384.augreg_in21k_ft_in1k\", pretrained=False,\n        )\n        self.vit.head = nn.Linear(self.vit.num_features, 1)\n\n    def forward(self, images):\n        return self.vit(images)\n\n\nclass FrozenCF:\n    def __init__(self, checkpoint_path, expected_sha256, threshold,\n                 expected_manifest_sha256, device=None):\n        if file_sha256(checkpoint_path) != expected_sha256:\n            raise RuntimeError(\"Checkpoint SHA-256 mismatch.\")\n        if not 0 <= threshold <= 1:\n            raise ValueError(\"Threshold must be in [0, 1].\")\n        self.threshold = float(threshold)\n        self.device = torch.device(device or (\"cuda\" if torch.cuda.is_available() else \"cpu\"))\n        if self.device.type == \"cuda\":\n            torch.backends.cuda.matmul.allow_tf32 = False\n            torch.backends.cudnn.allow_tf32 = False\n        self.model = CommunityForensicsDetector()\n        payload = torch.load(checkpoint_path, map_location=\"cpu\", weights_only=True)\n        if payload.get(\"stage\") != \"stage2_last_blocks\":\n            raise RuntimeError(\"Checkpoint is not Stage 2.\")\n        if payload.get(\"experiment_manifest_sha256\") != expected_manifest_sha256:\n            raise RuntimeError(\"Checkpoint belongs to a different experiment manifest.\")\n        self.best_val_auc = float(payload[\"best_val_auc\"])\n        self.model.load_state_dict(payload[\"model_state\"], strict=True)\n        self.model.requires_grad_(False)\n        self.model.float().eval().to(self.device)\n        self.preprocess = transforms.Compose([\n            transforms.Resize(440, interpolation=transforms.InterpolationMode.BILINEAR,\n                              antialias=True),\n            transforms.CenterCrop(384),\n            transforms.ToTensor(),\n            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),\n        ])\n\n    def predict(self, path):\n        with Image.open(path) as source:\n            source.seek(0)\n            image = ImageOps.exif_transpose(source).convert(\"RGB\")\n            tensor = self.preprocess(image).unsqueeze(0).to(self.device)\n        with torch.inference_mode():\n            logits = self.model(tensor).reshape(-1)\n            if logits.numel() != 1 or not torch.isfinite(logits).all():\n                raise RuntimeError(\"Invalid model output.\")\n            score = float(torch.sigmoid(logits)[0].cpu())\n            logit = float(logits[0].cpu())\n        return {\"logit\": logit, \"prob_fake\": score,\n                \"predicted_label\": int(score >= self.threshold)}\n"
import types
import shutil
import tempfile
import torch

runtime_versions = {
    name: importlib.metadata.version(name)
    for name in ("torch", "torchvision", "timm", "Pillow", "pandas", "numpy")
}
for name in ("timm", "Pillow"):
    require(runtime_versions[name] == experiment["libraries"][name],
            f"{name} version differs. Restart with a fresh runtime and run from the top.")

inference_module = types.ModuleType("ai_shield_cf_inference")
exec(compile(MODULE_SOURCE, "ai_shield_cf_inference.py", "exec"), inference_module.__dict__)
import PIL
require(PIL.__version__ == experiment["libraries"]["Pillow"],
        "Pillow was already imported at a different version. Restart the runtime.")

with tempfile.TemporaryDirectory(prefix="ai_shield_cf_") as temporary:
    snapshot = Path(temporary) / "best.pt"
    shutil.copyfile(CHECKPOINT, snapshot)
    detector = inference_module.FrozenCF(
        snapshot, checkpoint_sha256, DEFAULT_THRESHOLD, manifest_sha256,
    )
require(math.isclose(detector.best_val_auc, report["validation"]["finetuned"]["roc_auc"],
                     rel_tol=0, abs_tol=1e-12), "Checkpoint validation AUC differs.")
print("Loaded selected checkpoint on", detector.device)
print("Runtime:", runtime_versions)


## 6. Compare saved and standalone scores

For each image, compare both logits and sigmoid scores using the tolerances recorded before inference. Classifications at the default threshold must also match. Ground-truth labels only select both classes; this test asks whether predictions reproduce, not whether every prediction is correct.

The report is saved even on a score mismatch. If it fails, investigate the runtime, checkpoint and preprocessing. Do not loosen tolerances or switch fixtures just to obtain PASS.


In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_OUTPUT = FREEZE_ROOT / "verification_runs" / run_id
RUN_OUTPUT.mkdir(parents=True, exist_ok=False)
rows = []
for fixture in fixtures:
    row = dict(fixture)
    try:
        require(sha256_file(fixture["path"]) == fixture["sha256"],
                "Fixture changed after selection.")
        output = detector.predict(fixture["path"])
        row.update(output)
        row["prob_abs_error"] = abs(output["prob_fake"] - fixture["expected_prob_fake"])
        row["logit_abs_error"] = abs(output["logit"] - fixture["expected_logit"])
        row["prob_matches"] = bool(np.isclose(
            output["prob_fake"], fixture["expected_prob_fake"], atol=PROB_ATOL, rtol=PROB_RTOL))
        row["logit_matches"] = bool(np.isclose(
            output["logit"], fixture["expected_logit"], atol=LOGIT_ATOL, rtol=LOGIT_RTOL))
        row["expected_label"] = int(fixture["expected_prob_fake"] >= DEFAULT_THRESHOLD)
        row["label_matches"] = output["predicted_label"] == row["expected_label"]
        row["passed"] = row["prob_matches"] and row["logit_matches"] and row["label_matches"]
        row["error"] = ""
    except Exception as exc:
        row.update({"passed": False, "error": f"{type(exc).__name__}: {exc}"})
    rows.append(row)

comparison = pd.DataFrame(rows)
comparison.to_csv(RUN_OUTPUT / "prediction_comparison.csv", index=False)
all_passed = len(rows) == 2 * SAMPLES_PER_CLASS and all(row["passed"] for row in rows)
checkpoint_unchanged = sha256_file(CHECKPOINT) == checkpoint_sha256
all_passed = all_passed and checkpoint_unchanged
verification = {
    "schema_version": 1, "status": "PASS" if all_passed else "FAIL",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "detector_lock_sha256": sha256_file(LOCK_PATH),
    "fine_tuned_checkpoint_sha256": checkpoint_sha256,
    "checkpoint_unchanged_during_check": checkpoint_unchanged,
    "inference_source_sha256": hashlib.sha256(MODULE_SOURCE.encode("utf-8")).hexdigest(),
    "runtime_versions": runtime_versions,
    "device": str(detector.device),
    "device_name": (torch.cuda.get_device_name(detector.device)
                    if detector.device.type == "cuda" else "CPU"),
    "cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "comparison_policy": lock["comparison"],
    "checked": len(rows), "passed": sum(bool(row["passed"]) for row in rows),
    "results": rows,
    "scope": "Six validation fixtures; implementation parity, not a generalization benchmark.",
}
write_once(RUN_OUTPUT / "verification_report.json", json_text(verification))
display(comparison)
print("Verification:", verification["status"])
print("Report:", RUN_OUTPUT / "verification_report.json")
require(all_passed, "Inference parity failed. Read the report; no verified config was exported.")


## 7. Export the verified detector configuration

This cell rechecks the PASS evidence before exporting. Each run has its own directory, so a later failed run cannot replace a successful export. The configuration points to the existing checkpoint; it does not duplicate or change it.

The exported Python module can be reused by the provenance notebook or app. Moving to another device/runtime requires another parity check.


In [ ]:
require(verification["status"] == "PASS" and all_passed, "A passing check is required.")
require(sha256_file(CHECKPOINT) == checkpoint_sha256, "Checkpoint changed before export.")
require(sha256_file(LOCK_PATH) == verification["detector_lock_sha256"], "Lock file changed.")
require(hashlib.sha256(MODULE_SOURCE.encode("utf-8")).hexdigest() ==
        verification["inference_source_sha256"], "Inference source changed after verification.")

config = {
    **lock,
    "verification": {
        "status": "PASS", "report_path": str(RUN_OUTPUT / "verification_report.json"),
        "report_sha256": sha256_file(RUN_OUTPUT / "verification_report.json"),
        "runtime_versions": runtime_versions,
        "device": verification["device"], "checked": len(rows),
    },
    "inference_module": {
        "path": str(RUN_OUTPUT / "ai_shield_cf_inference.py"),
        "sha256": verification["inference_source_sha256"],
    },
}
write_once(RUN_OUTPUT / "detector_config.json", json_text(config))
write_once(RUN_OUTPUT / "ai_shield_cf_inference.py", MODULE_SOURCE)

print("PASS - frozen detector identity and standalone inference verified.")
print("Fine-tuned checkpoint SHA-256:", checkpoint_sha256)
print("Default threshold:", DEFAULT_THRESHOLD)
print("Configuration:", RUN_OUTPUT / "detector_config.json")
print("Reusable inference:", RUN_OUTPUT / "ai_shield_cf_inference.py")
print("Score comparison:", RUN_OUTPUT / "prediction_comparison.csv")


## What this establishes

- The exact fine-tuned checkpoint is identified by its own SHA-256.
- The default is the saved validation max-F1 threshold, with the alternative documented.
- Standalone inference reproduces six fixed validation examples within the recorded tolerances.
- Original source files remain unchanged.

The next notebook can add ExifTool and C2PA evidence around this frozen detector. Missing metadata will remain unknown and will not change the detector score.
